# Feature engineering — Create useful predictors from repayment history.
- Feature engineering — Create useful predictors from repayment history.

- Train/test splitting — Reserve unseen data to evaluate generalisation.

- Preprocessing — Prepare features using a scikit-learn pipeline.

- Logistic Regression — Train your first baseline machine learning model.

- Model evaluation — Examine confusion matrix, precision, recall and ROC-AUC.

- Random Forest — Compare a more flexible algorithm.

In [1]:
# Step 1 — Prepare the modelling dataset
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/credit_card_cleaned.csv")

print("Data shape:", df.shape)
display(df.head())

Data shape: (30000, 25)


,ID,LIMIT_BAL,SEX,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default,EDUCATION,MARRIAGE
0,1,20000.0,2,24,2,2,-1,-1,-2,-2,...,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1,2,1
1,2,120000.0,2,26,-1,2,0,0,0,2,...,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1,2,2
2,3,90000.0,2,34,0,0,0,0,0,0,...,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0,2,2
3,4,50000.0,2,37,0,0,0,0,0,0,...,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0,2,1
4,5,50000.0,1,57,-1,0,-1,0,0,0,...,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0,2,1


In [2]:
# Step 2 — Separate features and target, then split
from sklearn.model_selection import train_test_split

x = df.drop(columns=["default", "ID"]) # create a new DataFrame called X containing all columns from df except default and ID
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Training default rate:", y_train.mean())
print("Testing default rate:", y_test.mean())

Training: (24000, 23)
Testing: (6000, 23)
Training default rate: 0.22120833333333334
Testing default rate: 0.22116666666666668


In [5]:
# Step 3 — Feature engineering
# Feature engineering means creating new variables that may help the model identify patterns.

pay_cols = [
    "PAY_1", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

def add_features(x):
    x=x.copy()

    # Number of recorded months with a positive delay code
    x["months_with_delay"] = (
        x[pay_cols] > 0
    ).sum(axis=1)

    # Highest positive delay code across six months
    x["max_delay"] = (
        x[pay_cols].clip(lower=0).max(axis=1)
    )

    return x
X_train = add_features(X_train)
X_test = add_features(X_test)

In [6]:
# Step 4 — Train your first Logistic Regression model

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([ # A Pipeline connects multiple machine learning steps so they execute automatically in the correct order.
    ("imputer", SimpleImputer(strategy="median")), # Sometimes dataset contains missing values (NaN). SimpleImputer calculates the median of each numerical column and replaces missing values.
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [7]:
# Step 5 — Evaluate the first model
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       0.83      0.95      0.89      4673
           1       0.65      0.31      0.42      1327

    accuracy                           0.81      6000
   macro avg       0.74      0.63      0.65      6000
weighted avg       0.79      0.81      0.78      6000

Confusion matrix:
[[4458  215]
 [ 922  405]]
ROC-AUC: 0.7469296835982043
PR-AUC: 0.5102658533298868
